# Notebook 1 — Setup & Dataset
**Project:** Sensitivity of Diffusion Models to Noise Distribution Choice

This notebook handles:
- Installing dependencies
- Mounting Google Drive for checkpoint persistence
- Loading and exploring the MNIST dataset
- Defining the project config (shared across all notebooks)
- Defining the base `ForwardDiffusion` class (subclassed in Notebook 2)

## 1. Install & Imports

In [ ]:
!pip install -q torch torchvision tqdm matplotlib numpy

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Mount Google Drive
All checkpoints and logs will be saved here so they survive session timeouts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project directory structure
PROJECT_DIR = '/content/drive/MyDrive/diffusion_noise_project'
for subdir in ['checkpoints/gaussian', 'checkpoints/uniform', 'checkpoints/laplace',
               'logs', 'samples', 'figures']:
    os.makedirs(os.path.join(PROJECT_DIR, subdir), exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print("Directory structure created.")

## 3. Global Config
All hyperparameters in one place. This dict will be copied verbatim into training notebooks — the only thing that changes is `noise_type`.

In [ ]:
CONFIG = {
    # Data
    'dataset': 'MNIST',
    'image_size': 28,
    'channels': 1,
    'batch_size': 128,

    # Diffusion process
    'T': 1000,               # number of diffusion steps
    'beta_start': 1e-4,      # noise schedule start
    'beta_end': 0.02,        # noise schedule end
    'schedule': 'linear',    # 'linear' or 'cosine'

    # Training
    'num_epochs': 30,
    'lr': 2e-4,
    'optimizer': 'adamw',
    'weight_decay': 1e-4,
    'num_workers': 2,

    # Model
    'model_channels': 64,    # base channel count in U-Net
    'channel_mults': (1, 2, 4),

    # Logging
    'log_every': 100,        # steps
    'save_every': 5,         # epochs

    # Reproducibility
    'seed': 42,
}

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(CONFIG['seed'])
print("Config set. Seed fixed.")
print(CONFIG)

## 4. Load MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # scale to [-1, 1]
])

train_dataset = torchvision.datasets.MNIST(
    root='/content/data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root='/content/data', train=False, download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=True
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers']
)

print(f"Train samples: {len(train_dataset):,}")
print(f"Test samples:  {len(test_dataset):,}")
print(f"Batches per epoch: {len(train_loader)}")

## 5. Explore the Dataset

In [ ]:
def show_samples(dataset, n=32, title='MNIST samples'):
    indices = np.random.choice(len(dataset), n, replace=False)
    images = torch.stack([dataset[i][0] for i in indices])
    # Unnormalise for display
    images = (images * 0.5 + 0.5).clamp(0, 1)

    grid = torchvision.utils.make_grid(images, nrow=8, padding=2)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    ax.axis('off')
    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_DIR, 'figures', 'mnist_samples.png'), dpi=150, bbox_inches='tight')
    plt.show()

show_samples(train_dataset)

In [ ]:
# Pixel value distribution
sample_batch, _ = next(iter(train_loader))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(sample_batch.numpy().flatten(), bins=100, color='steelblue', edgecolor='none')
axes[0].set_title('Pixel value distribution (normalised to [-1,1])')
axes[0].set_xlabel('Pixel value')
axes[0].set_ylabel('Count')

axes[1].imshow(sample_batch[0, 0].numpy(), cmap='gray')
axes[1].set_title(f'Single sample — shape: {tuple(sample_batch[0].shape)}')
axes[1].axis('off')

plt.tight_layout()
plt.show()
print(f"Batch shape: {sample_batch.shape}")
print(f"Value range: [{sample_batch.min():.3f}, {sample_batch.max():.3f}]")

## 6. Beta Schedule
Define the noise schedule — shared across all experiments.

In [ ]:
def make_beta_schedule(schedule, T, beta_start, beta_end):
    """Returns betas, alphas, and cumulative alphas for the diffusion process."""
    if schedule == 'linear':
        betas = torch.linspace(beta_start, beta_end, T)
    elif schedule == 'cosine':
        steps = T + 1
        x = torch.linspace(0, T, steps)
        alphas_cumprod = torch.cos(((x / T) + 0.008) / 1.008 * torch.pi / 2) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        betas = betas.clamp(0, 0.999)
    else:
        raise ValueError(f"Unknown schedule: {schedule}")

    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    alphas_cumprod_prev = torch.cat([torch.tensor([1.0]), alphas_cumprod[:-1]])

    return {
        'betas': betas,
        'alphas': alphas,
        'alphas_cumprod': alphas_cumprod,
        'alphas_cumprod_prev': alphas_cumprod_prev,
        'sqrt_alphas_cumprod': alphas_cumprod.sqrt(),
        'sqrt_one_minus_alphas_cumprod': (1 - alphas_cumprod).sqrt(),
    }

schedule = make_beta_schedule(
    CONFIG['schedule'], CONFIG['T'], CONFIG['beta_start'], CONFIG['beta_end']
)

# Visualise the schedule
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
t = np.arange(CONFIG['T'])

axes[0].plot(t, schedule['betas'].numpy())
axes[0].set_title('Beta schedule')
axes[0].set_xlabel('Timestep t')

axes[1].plot(t, schedule['alphas_cumprod'].numpy())
axes[1].set_title('Cumulative alpha (signal retention)')
axes[1].set_xlabel('Timestep t')

axes[2].plot(t, schedule['sqrt_alphas_cumprod'].numpy(), label='sqrt(ā_t)')
axes[2].plot(t, schedule['sqrt_one_minus_alphas_cumprod'].numpy(), label='sqrt(1-ā_t)')
axes[2].set_title('Signal vs noise coefficient')
axes[2].set_xlabel('Timestep t')
axes[2].legend()

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'figures', 'beta_schedule.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Base ForwardDiffusion Class
Subclassed in Notebook 2 for each noise distribution.

In [ ]:
class ForwardDiffusion:
    """
    Base class for the forward diffusion process.
    Subclasses override `sample_noise` to change the distribution.

    The forward process is:
        x_t = sqrt(ā_t) * x_0 + sqrt(1 - ā_t) * eps
    where eps ~ noise_distribution(0, I).

    All distributions are standardised to have zero mean and unit variance,
    so comparisons are fair.
    """

    def __init__(self, schedule, device):
        self.device = device
        self.T = len(schedule['betas'])
        # Move all tensors to device
        for k, v in schedule.items():
            setattr(self, k, v.to(device))

    def sample_noise(self, shape):
        """Override in subclasses. Must return zero-mean, unit-variance noise."""
        raise NotImplementedError

    def sample_timestep(self, batch_size):
        """Uniformly sample a timestep for each item in the batch."""
        return torch.randint(0, self.T, (batch_size,), device=self.device).long()

    def q_sample(self, x_0, t):
        """
        Sample x_t from q(x_t | x_0) using the reparameterisation trick.
        Returns both x_t and the noise eps (needed for the training objective).
        """
        eps = self.sample_noise(x_0.shape)

        sqrt_alpha_bar = self.sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_one_minus_alpha_bar = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]

        x_t = sqrt_alpha_bar * x_0 + sqrt_one_minus_alpha_bar * eps
        return x_t, eps

    def noise_stats(self, n_samples=100000):
        """Verify that the noise is correctly standardised."""
        noise = self.sample_noise((n_samples, 1, 1, 1))
        mean = noise.mean().item()
        std = noise.std().item()
        return {'mean': mean, 'std': std}


print("ForwardDiffusion base class defined.")
print("Subclasses need to implement: sample_noise(shape) -> tensor")

## 8. Sanity Check — Helper Utilities

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def unnormalise(x):
    """Undo the [-1,1] normalisation for display."""
    return (x * 0.5 + 0.5).clamp(0, 1)

def save_config(config, path):
    import json
    with open(path, 'w') as f:
        json.dump(config, f, indent=2)

save_config(CONFIG, os.path.join(PROJECT_DIR, 'config.json'))
print("Config saved to Drive.")
print("\nNotebook 1 complete. Proceed to Notebook 2.")